In [1]:
import json
from collections import defaultdict
from pathlib import Path

import polars as pl

import src.social_groups.polars_columns as plc
from social_groups.analysis.polars_transformations.apply_parsing_and_group_decision import (
    apply_parsing_and_group_decision,
)
from social_groups.analyzer.io_operations import read_hydra_config
from social_groups.directories import TRACK_FILE_NAME, TRACK_FILE_NAME_COMPRESSED
from social_groups.general.tracking import TrackEntry, iter_jsonl_zst
from social_groups.polars_values import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)
from social_groups.trialrunner.data_connectors.mmlu_pro_subset import (
    MMLUProSubsetConnector,
)
from social_groups.trialrunner.utils.hydra_config import MainConfig

%load_ext autoreload
%autoreload 2


Using Phoenix Cache located at /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/phoenix/CACHE.


In [2]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

DIR = Path(
    "/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/multirun/todo/tribal_council/2026-02-28-16-07-58")

In [3]:
data_connector = MMLUProSubsetConnector()

input_data = pl.DataFrame([p.model_dump() | {"question": data_connector.prepare_example(p).question} for p in
                           data_connector.iterate_data()])

In [4]:
data = defaultdict(list)
info: dict[str, MainConfig] = {}
for i, project_path in enumerate(sorted(DIR.iterdir())):
    if not project_path.is_dir():
        continue
    run_id = i

    run_config = read_hydra_config(project_path)
    info[project_path] = run_config
    # run_meta_info = read_meta_config(project_path)

    if (project_path / TRACK_FILE_NAME_COMPRESSED).exists():

        for item in iter_jsonl_zst(project_path / TRACK_FILE_NAME_COMPRESSED):
            data[project_path].append(TrackEntry.model_validate(item))
    else:
        with (project_path / TRACK_FILE_NAME).open("rb") as f:
            for line in f.readlines():
                data[project_path].append(TrackEntry.model_validate(json.loads(line)))

df = pl.DataFrame(
    [{"question": (x := d.model_dump())["input"]["question"]} | {"project_path": project.name,
                                                                 "num_different_proposals": info[
                                                                     project].experiment.strategy.configuration.num_different_proposals,
                                                                 "council_size": info[
                                                                     project].experiment.strategy.configuration.council_size,
                                                                 "proposal_agent":
                                                                     info[
                                                                         project].experiment.strategy.configuration.proposal_agent.backend.model_name,
                                                                 "council_agent":
                                                                     info[
                                                                         project].experiment.strategy.configuration.council_agent.backend.model_name} | {
         a: b for a, b in
         x["output"].items()}
     for project, datas in data.items() for d in datas],
).drop("used_input_tokens", "used_output_tokens", "final_answer", "history")

In [5]:
tribal_council_results = (
    apply_parsing_and_group_decision(
        df.join(input_data, on="question").drop("src", "category", "cot_content").rename(
            {"answer": "answer_string"}), parser, comparer, group_reply)
).with_columns(pl.col("proposal_agent", "council_agent").replace(MODEL_NAME_TO_LETTER_MAPPING))


In [6]:
tribal_council_results

question,project_path,num_different_proposals,council_size,proposal_agent,council_agent,answers_at_beginning,answers_at_end,question_id,answer_index,answer_string,options,___parsed_individual_answers_before___,___parsed_individual_answers_after___,___parsed_combined_answers_before___,___parsed_combined_answers_after___,is_correct
str,str,i64,i64,str,str,list[str],list[str],i64,i64,str,list[str],list[str],list[str],str,str,bool
"""Q: In 2018, about how many chi…","""tribal_council_0""",2,2,"""L""","""L""","[""D"", ""I""]","[""D"", ""D""]",5681,6,"""G""","[""690 billion"", ""1 trillion"", … ""25 billion""]","[""D"", ""I""]","[""D"", ""D""]","""___different_votes___""","""D""",false
"""Q: Kirkwood gaps are observed …","""tribal_council_0""",2,2,"""L""","""L""","[""D"", ""G""]","[""D: asteroids would orbit with a period twice that of Jupiter"", ""D: asteroids would orbit with a period twice that of Jupiter""]",10297,5,"""F""","[""asteroids would orbit with a period that is three times that of Jupiter"", ""asteroids would orbit with a period three times that of Mars"", … ""asteroids would orbit with a period that is equal to that of Jupiter""]","[""D"", ""G""]","[""D"", ""D""]","""___different_votes___""","""D""",false
"""Q: A 2-month-old female is bro…","""tribal_council_0""",2,2,"""L""","""L""","[""A"", ""D""]","[""(A) any chronic conditions and (D) allergy to eggs"", ""(A) any chronic conditions and (D) allergy to eggs""]",6116,5,"""F""","[""any chronic conditions"", ""dietary habits"", … ""history of heart disease""]","[""A"", ""D""]","[""A"", ""A""]","""___different_votes___""","""A""",false
"""Q: A wealthy woman often wore …","""tribal_council_0""",2,2,"""L""","""L""","[""A"", ""F""]","[""A"", ""A""]",1945,4,"""E""","[""Theft."", ""Criminal mischief."", … ""Fraud.""]","[""A"", ""F""]","[""A"", ""A""]","""___different_votes___""","""A""",false
"""Q: What stable isotope is comm…","""tribal_council_0""",2,2,"""L""","""L""","[""C"", ""J""]","[""J"", ""J""]",6622,9,"""J""","[""Nitrogen 14"", ""Oxygen 16"", … ""Deuterium""]","[""C"", ""J""]","[""J"", ""J""]","""___different_votes___""","""J""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q: In a given economy, househo…","""tribal_council_9""",2,3,"""L""","""L""","[""A""]",[],7140,0,"""A""","[""$18.07 billion"", ""$12.5 billion"", … ""$17 billion""]","[""A""]",[],"""A""","""___all_votes_invalid___""",false
"""Q: Camera lenses usually conta…","""tribal_council_9""",2,3,"""L""","""L""","[""F"", ""D""]","[""F"", ""F"", ""F: 2 mm""]",10353,7,"""H""","[""8 mm"", ""10 mm"", … ""5 mm""]","[""F"", ""D""]","[""F"", ""F"", ""F""]","""___different_votes___""","""F""",false
"""Q: Of the following, the best …","""tribal_council_9""",2,3,"""L""","""L""","[""D"", ""J""]","[""D: diffuse the responsibility among all members of the community"", ""D: diffuse the responsibility among all members of the community"", ""D""]",2089,4,"""E""","[""decrease the number of individuals in the community"", ""implement strict punishments for non-helping behavior"", … ""initiate programs to raise the self-esteem of community members""]","[""D"", ""J""]","[""D"", ""D"", ""D""]","""___different_votes___""","""D""",false


In [7]:
tribal_council_results.group_by("project_path", "num_different_proposals", "council_size", "proposal_agent",
                                "council_agent").agg(
    pl.col(plc.is_correct).mean().alias(plc.accuracy), available_data=pl.len() / 100).sort("accuracy", descending=True)

project_path,num_different_proposals,council_size,proposal_agent,council_agent,accuracy,available_data
str,i64,i64,str,str,f64,f64
"""tribal_council_15""",2,3,"""H""","""L""",0.73,1.0
"""tribal_council_35""",3,3,"""H""","""H""",0.714286,0.21
"""tribal_council_7""",2,2,"""H""","""M""",0.71,1.0
"""tribal_council_26""",3,2,"""H""","""H""",0.69697,0.66
"""tribal_council_8""",2,2,"""H""","""H""",0.69,1.0
…,…,…,…,…,…,…
"""tribal_council_29""",3,3,"""L""","""H""",0.29,1.0
"""tribal_council_19""",3,2,"""L""","""M""",0.28,1.0
"""tribal_council_20""",3,2,"""L""","""H""",0.28,1.0


In [8]:
data[Path(
    '/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/multirun/todo/tribal_council/2026-02-28-16-07-58/tribal_council_35')][
    0].phoenix_span_info

PhoenixExampleHandle(trace_id_hex='45b83f1e1b5928202295ca2edbb5d68a', span_id_hex='04392656cf7083f0', captured_span_ids=['04392656cf7083f0', 'b68b7597088278b2', 'dca257fa52977cea', '3426f2e8898f4412', '8006663f57dc7c0c', '6b88fc8106fd0842', 'f6830e602b3f2d4b', 'acb6250a8b0998a0', '8086fbe3d62d5a0f', '0ec62d2c18545575', '79f3d21c456dbf9e', '3d43e2528b69afdc', 'bb1350d578545c3a', '1bd7edecfb209e11', '6c0f49fb22470177', 'b1bd078ed1b9fff1', '58ae49a354d1aad3', 'd14497bf79348ea3', 'd1f00033b3251395', '91b2bc8d0de3f193', 'f13f0426d6f6b117', '985f8c8dcb1e09fe', 'b0af017c8561916a', '348ac362c79d29ee', '313d16f3d018d275', 'fbd3075a87db825c', 'ac9a2fbaea23e86d', '81531721211d62b0', '52bc7360a96bc5c9', 'ae53d082643633ea', 'ebd0777fc910688c', '935e888563d30a3f', 'e535797bd367aa2a', '55f55c5c1552679e', '7697d9dc436f13b6', '0c928f2a06ec8fa3', 'd2bf3219360fe2c4', '29c69d38aba76d00', '933b89aaa9fd10b0', '8a303cf0ff3dc8e0', '978f9669b0c75fbd', '2ec37af67c9d450a'])

In [9]:
# from social_groups.analyzer.utils import get_span_attributes

# get_span_attributes(span_ids=['04392656cf7083f0'], phoenix_graphql_endpoint="http://localhost:6006/graphql")

In [10]:
from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_decision_scheme import calculate_decision_scheme

for group in tribal_council_results.group_by("project_path", "num_different_proposals", "council_size",
                                             "proposal_agent",
                                             "council_agent"):
    print(group[0], "available data: ", group[1].shape[0])
    print(calculate_decision_scheme(
        group[1],
        AnalysisColumn.parsed_individual_answers_before.value,
        AnalysisColumn.parsed_individual_answers_after.value,
        "answer_string",
        group_reply,
        comparer,
    ).select("Correct Members Beginning", "correct", "incorrect"))

('tribal_council_3', 2, 2, 'M', 'L') available data:  100
shape: (3, 3)
┌───────────────────────────┬──────────┬───────────┐
│ Correct Members Beginning ┆ correct  ┆ incorrect │
│ ---                       ┆ ---      ┆ ---       │
│ u32                       ┆ f64      ┆ f64       │
╞═══════════════════════════╪══════════╪═══════════╡
│ 2                         ┆ 0.961538 ┆ 0.038462  │
│ 1                         ┆ 0.75     ┆ 0.25      │
│ 0                         ┆ 0.03125  ┆ 0.96875   │
└───────────────────────────┴──────────┴───────────┘
('tribal_council_9', 2, 3, 'L', 'L') available data:  100
shape: (3, 3)
┌───────────────────────────┬──────────┬───────────┐
│ Correct Members Beginning ┆ correct  ┆ incorrect │
│ ---                       ┆ ---      ┆ ---       │
│ u32                       ┆ f64      ┆ f64       │
╞═══════════════════════════╪══════════╪═══════════╡
│ 2                         ┆ 0.95     ┆ 0.05      │
│ 1                         ┆ 0.393939 ┆ 0.606061  │
│ 0     

In [11]:
COUNCIL_NUMBER = 15

In [12]:
from social_groups.reporting.group_decision_scheme import (
    calculate_extended_decision_scheme,
)

dec_scheme = calculate_extended_decision_scheme(
    tribal_council_results.filter(pl.col("project_path") == f"tribal_council_{COUNCIL_NUMBER}"),
    AnalysisColumn.parsed_individual_answers_before.value,
    AnalysisColumn.parsed_individual_answers_after.value,
    "answer_string",
    group_reply,
    comparer,
)
dec_scheme

Correct Members Beginning,Correct Members End,correct,occurrences,incorrect
u32,u32,f64,u32,f64
2,3,1.0,57,0.0
1,3,1.0,13,0.0
1,0,0.0,8,1.0
0,3,1.0,3,0.0
0,0,0.0,19,1.0


In [13]:
from social_groups.reporting.plots.decision_scheme_extended import (
    make_decision_scheme_extended_plot,
)

make_decision_scheme_extended_plot(
    dec_scheme, title=f"Group {COUNCIL_NUMBER}"
).show(width=800, height=600)

In [14]:
all_cases = list(
    sorted(
        set(dec_scheme["Correct Members Beginning"].unique()).union(
            set(dec_scheme["Correct Members End"].unique())
        )
    )
)

all_cases

[0, 1, 2, 3]

In [15]:
left_groups = [f" {x}☑ (Start)" for x in all_cases]
right_groups = [f" {x}☑ (End)" for x in all_cases]
left_groups, right_groups

([' 0☑ (Start)', ' 1☑ (Start)', ' 2☑ (Start)', ' 3☑ (Start)'],
 [' 0☑ (End)', ' 1☑ (End)', ' 2☑ (End)', ' 3☑ (End)'])

In [16]:
dec_scheme

Correct Members Beginning,Correct Members End,correct,occurrences,incorrect
u32,u32,f64,u32,f64
2,3,1.0,57,0.0
1,3,1.0,13,0.0
1,0,0.0,8,1.0
0,3,1.0,3,0.0
0,0,0.0,19,1.0


In [17]:
correct_at_end_given_correct_start_members = (
    pl.DataFrame({"Correct Members Beginning": all_cases})
    .join(
        dec_scheme.group_by("Correct Members Beginning")
        .agg(
            correct=(
                    pl.col("correct").dot(pl.col("occurrences")) / pl.sum("occurrences")
            ).mean()
        )
        .select("correct", "Correct Members Beginning"),
        on="Correct Members Beginning",
        how="left",
    )
    .with_columns(pl.col("correct").fill_null(0))
    .sort("Correct Members Beginning", descending=True)
    .select("correct")
    .to_numpy()
    .reshape(len(right_groups), 1)
)

correct_at_end_given_correct_start_members

array([[0.        ],
       [1.        ],
       [0.61904762],
       [0.13636364]])

In [18]:
dec_scheme.select(
    "Correct Members Beginning", "Correct Members End", "occurrences"
)

Correct Members Beginning,Correct Members End,occurrences
u32,u32,u32
2,3,57
1,3,13
1,0,8
0,3,3
0,0,19


In [19]:
dec_scheme.schema

Schema([('Correct Members Beginning', UInt32),
        ('Correct Members End', UInt32),
        ('correct', Float64),
        ('occurrences', UInt32),
        ('incorrect', Float64)])

In [20]:
from itertools import repeat
import numpy as np

links = (
    pl.DataFrame({"Correct Members Beginning": list(y for x in repeat(all_cases, len(all_cases)) for y in x),
                  "Correct Members End": [y for x in all_cases for y in repeat(x, len(all_cases))]},
                 schema={"Correct Members Beginning": pl.datatypes.UInt32, "Correct Members End": pl.datatypes.UInt32})
    .join(
        dec_scheme,
        on=["Correct Members Beginning", "Correct Members End"],
        how="left",
    ).with_columns(pl.col("occurrences").fill_null(0))
    .select(
        "Correct Members Beginning", "Correct Members End", "occurrences"
    ).map_rows(lambda r: np.array(r))["map"]
    .to_list()
)

links

[[0, 0, 19],
 [1, 0, 8],
 [2, 0, 0],
 [3, 0, 0],
 [0, 1, 0],
 [1, 1, 0],
 [2, 1, 0],
 [3, 1, 0],
 [0, 2, 0],
 [1, 2, 0],
 [2, 2, 0],
 [3, 2, 0],
 [0, 3, 3],
 [1, 3, 13],
 [2, 3, 57],
 [3, 3, 0]]

In [21]:
nL = len(left_groups)
nL

4

In [22]:
source = [s for s, t, v in links]
target = [nL + t for s, t, v in links]
value = [v for s, t, v in links]

In [23]:
source, target, value

([0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3],
 [4, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6, 6, 7, 7, 7, 7],
 [19, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 13, 57, 0])

In [24]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from pyfonts import load_google_font


def hex_to_rgba(hex_color, a=0.22):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{a})"


make_subplots(
    rows=1,
    cols=3,
    column_widths=[0.4, 2, 0.4],
    specs=[[{"type": "heatmap"}, {"type": "sankey"}, {"type": "heatmap"}]],
    horizontal_spacing=0.01,
    subplot_titles=[
        "Correct Group Dec.<br> (p. No. corr. @ start)",
        "TITLE" + "<br> ",
        "",
    ],
).add_trace(
    go.Sankey(
        arrangement="snap",
        node=dict(
            label=left_groups + right_groups,
            pad=20,
            thickness=16,
            color=["#A3B1C6", "#B9A3C9", "#323647", "#D4A35C", "#6C7A89"],
            line=dict(color="rgba(0,0,0,0.3)", width=0.6),
            # Change the order ot the nodes ...
            x=[0.01 for _ in left_groups] + [0.99 for _ in right_groups],
            y=np.cumsum([0.17 for _ in left_groups]).tolist()[::-1] * 2,
        ),
        link=dict(
            source=source,
            target=target,
            value=value,
            color=[hex_to_rgba(["#A3B1C6", "#B9A3C9", "#323647", "#D4A35C", "#6C7A89"][s], 0.22) for s in source],
        ),
        textfont=dict(
            family=load_google_font("Libre Baskerville", weight="bold").get_name(),
            size=12,
            color="rgb(34,32,32)",
            shadow="none",
        ),
    ),
    row=1,
    col=2,
)